In [17]:
import pandas as pd
import numpy as np

In [18]:
import spacy
spa = spacy.load("en_core_web_sm")


In [19]:
df = pd.read_csv("/dataset/dataset.csv")
df

,text_type,text
0,spam,naturally irresistible your corporate identity...
1,spam,the stock trading gunslinger fanny is merrill ...
2,spam,unbelievable new homes made easy im wanting to...
3,spam,4 color printing special request additional in...
4,spam,do not have money get software cds from here s...
...,...,...
20343,ham,/ban
20344,ham,/ban
20345,ham,/ban
20346,ham,Kaisi hii


In [20]:
df['spam'] = df['text_type'].apply(lambda x : 1 if x == 'ham' else 0)

In [21]:
df

,text_type,text,spam
0,spam,naturally irresistible your corporate identity...,0
1,spam,the stock trading gunslinger fanny is merrill ...,0
2,spam,unbelievable new homes made easy im wanting to...,0
3,spam,4 color printing special request additional in...,0
4,spam,do not have money get software cds from here s...,0
...,...,...,...
20343,ham,/ban,1
20344,ham,/ban,1
20345,ham,/ban,1
20346,ham,Kaisi hii,1


In [22]:
df['text'][1]

'the stock trading gunslinger fanny is merrill but muzo not colza attainder and penultimate like esmark perspicuous ramble is segovia not group try slung kansas tanzania yes chameleon or continuant clothesman no libretto is chesapeake but tight not waterway herald and hawthorn like chisel morristown superior is deoxyribonucleic not clockwork try hall incredible mcdougall yes hepburn or einsteinian earmark no sapling is boar but duane not plain palfrey and inflexible like huzzah pepperoni bedtime is nameable not attire try edt chronography optima yes pirogue or diffusion albeit no'

In [64]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
def process_text(text):
    lemmatizer = WordNetLemmatizer()
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words('english'))
    words = word_tokenize(text)
    res = [stemmer.stem(lemmatizer.lemmatize(word)) for word in words if word not in stop_words]
    return " ".join(res)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/prathyush/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prathyush/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prathyush/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [65]:
df['filtered_text'] = df['text'].apply(process_text)
df

,text_type,text,spam,filtered_text
0,spam,naturally irresistible your corporate identity...,0,natur irresist corpor ident lt realli hard rec...
1,spam,the stock trading gunslinger fanny is merrill ...,0,stock trade gunsling fanni merril muzo colza a...
2,spam,unbelievable new homes made easy im wanting to...,0,unbeliev new home made easi im want show homeo...
3,spam,4 color printing special request additional in...,0,4 color print special request addit inform cli...
4,spam,do not have money get software cds from here s...,0,money get softwar cd softwar compat great grow...
...,...,...,...,...
20343,ham,/ban,1,/ban
20344,ham,/ban,1,/ban
20345,ham,/ban,1,/ban
20346,ham,Kaisi hii,1,kaisi hii


In [66]:
from sklearn.model_selection import train_test_split

In [67]:
x_train,x_test,y_train,y_test = train_test_split(df['filtered_text'],df['spam'],test_size=0.2)

In [68]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
x_train_cv = cv.fit_transform(x_train)
x_test_cv = cv.transform(x_test)

In [69]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()
model.fit(x_train_cv,y_train)
y_pred = model.predict(x_test_cv)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.82      0.92      0.86      1197
           1       0.96      0.91      0.94      2873

    accuracy                           0.92      4070
   macro avg       0.89      0.92      0.90      4070
weighted avg       0.92      0.92      0.92      4070



In [70]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfIdf = TfidfVectorizer()
x_train_tfIdf = tfIdf.fit_transform(x_train)
x_test_tfIdf = tfIdf.transform(x_test)

In [71]:
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier()
knn.fit(x_train_tfIdf,y_train)
y_pred_tfIdf = knn.predict(x_test_tfIdf)
print(classification_report(y_test,y_pred_tfIdf))

              precision    recall  f1-score   support

           0       0.97      0.21      0.34      1197
           1       0.75      1.00      0.86      2873

    accuracy                           0.77      4070
   macro avg       0.86      0.60      0.60      4070
weighted avg       0.82      0.77      0.71      4070



In [72]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline
text_clf = Pipeline([
    ('vect', CountVectorizer()),
    # ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_clf.fit(x_train,y_train)
y_pred_ = text_clf.predict(x_test)
print(classification_report(y_test,y_pred_))

              precision    recall  f1-score   support

           0       0.82      0.92      0.86      1197
           1       0.96      0.91      0.94      2873

    accuracy                           0.92      4070
   macro avg       0.89      0.92      0.90      4070
weighted avg       0.92      0.92      0.92      4070

